# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnologico de Monterrey
#### Prof Luis Eduardo Falcon Morales

### **Actividad en Equipos Semanas - RAG Chatbot (v2 - Anti-Hallucination)**


Construct a QA Bot that Leverages LangChain and LLMs to Answer Questions from Loaded Documents
Estimated time needed: 60 minutes

In this project, you will construct a question-answering (QA) bot. This bot will leverage LangChain and a large language model (LLM) to answer questions based on content from loaded PDF documents. To build a fully functional QA system, you'll combine various components, including document loaders, text splitters, embedding models, vector databases, retrievers, and Gradio as the front-end interface.

Imagine you're tasked with creating an intelligent assistant that can quickly and accurately respond to queries based on a company's extensive library of PDF documents. This could be anything from legal documents to technical manuals. Manually searching through these documents would be time-consuming and inefficient.

* **Nombres y matriculas:**

  *   Jose Angel Barajas A01797221
  *   Elemento de lista
  *   Elemento de lista

* **Numero de Equipo:**


## 🛡️ v2 Upgrades — Anti-Hallucination

| # | Upgrade | Purpose |
|---|---------|---------|
| 1 | **Anti-hallucination system prompt** | Explicitly forbids guessing, demands citations per sentence |
| 2 | **Similarity threshold filtering** | Drops chunks below cosine similarity cutoff |
| 3 | **Confidence scoring** | Reports avg similarity of retrieved chunks |
| 4 | **Inline citations in answers** | Model outputs `[doc:i]` markers, reference list shown |
| 5 | **Conversation summary memory** | Summarized history prevents drift/fabrication |
| 6 | **Source chunks displayed in UI** | Grounding verification for every answer |
| 7 | "I don't know" fallback | If no chunks pass threshold, refuses to answer instead of guessing |

🧩 Step 1 – Install the required packages

In your VS Code notebook, create a first cell and run:

In [ ]:
!pip install langchain
!pip install langchain-classic 
!pip install langchain-community
!pip install langchain-openai
!pip install langchain-huggingface
!pip install chromadb
!pip install pypdf
!pip install sentence-transformers
!pip install gradio

⚙️ Step 2 – Imports

In [ ]:
## UPGRADE v2: Added imports for anti-hallucination features

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

## UPGRADE v2: Conversation chain + summary memory instead of RetrievalQA
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationSummaryMemory
from langchain_core.prompts import PromptTemplate

import gradio as gr
import os

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

Step 3 – LLM config (points to LM Studio)

**Note:** Same LM Studio config as v1. No changes to model or base URL.

In [ ]:
# LLM config for LM Studio
LMSTUDIO_BASE_URL = "http://100.111.50.52:1234/v1"   # note the /v1
LMSTUDIO_MODEL    = "qwen2.5-coder-7b-instruct"    # from LM Studio "API identifier"

def get_llm():
    llm = ChatOpenAI(
        base_url=LMSTUDIO_BASE_URL,
        api_key="not-needed",     # LM Studio doesn't check this, but parameter is required
        model=LMSTUDIO_MODEL,
        temperature=0.3,          ## UPGRADE v2: Lower temp (0.3) reduces random hallucinated content
        max_tokens=2048,          ## UPGRADE v2: Increased from 1024 for more complete grounded answers
    )
    return llm

In [ ]:
## UPGRADE v2: Test LLM connection
llm_test = get_llm()
resp = llm_test.invoke("Dame una respuesta corta en espanol diciendo que la conexion con LM Studio funciona.")
print(resp.content)

Step 4 – Document loader

In [ ]:
## UPGRADE v2: Added doc_id tracking per page for source attribution
def document_loader(file_path: str):
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    # Tag each page with its source filename for citation
    source_name = os.path.basename(file_path)
    for i, doc in enumerate(docs):
        doc.metadata["source_file"] = source_name
        doc.metadata["page_index"] = i
    return docs

Step 5 – Text splitter

In [ ]:
# UPGRADE v2: No change to splitter (chunk_size=1000, overlap=150 as in v1)

In [ ]:
def text_splitter(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        length_function=len,
    )
    chunks = splitter.split_documents(docs)
    return chunks

Step 6 – Embeddings + VectorDB

**Note:** Same embedding model and vector store as v1. No changes.

In [ ]:
def embedding_model():
    # SAME as v1: kept unchanged per instructions
    return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def vector_database(chunks):
    embed = embedding_model()
    vectordb = Chroma.from_documents(documents=chunks, embedding=embed)
    return vectordb

## 🛡️ Step 7 – Retriever with Similarity Threshold

**Similarity threshold** is key for reducing hallucinations: chunks that are semantically unrelated to the query get filtered out before they can influence the LLM's answer.

In [ ]:
## UPGRADE v2: New retriever with similarity threshold and confidence scoring

## UPGRADE v2: CHANGED from v1: build_retriever now returns a dict with both
## the retriever and the vector DB, so we can compute confidence scores.

SIMILARITY_THRESHOLD = 0.35  ## UPGRADE v2: Filter out chunks below this cosine similarity
TOP_K = 5                     ## UPGRADE v2: Max chunks to retrieve

def build_retriever(file_paths):
    """
    Builds a retriever from the given PDF files.
    
    Returns a dict:
      - "retriever": the Chroma retriever (top_k)
      - "vectorstore": the Chroma vectorstore (needed for similarity search)
    """
    all_docs = []
    for fp in file_paths:
        docs = document_loader(fp)
        all_docs.extend(docs)
    chunks = text_splitter(all_docs)
    vectordb = vector_database(chunks)
    
    retriever = vectordb.as_retriever(
        search_type="similarity",
        search_kwargs={"k": TOP_K}  ## UPGRADE v2: configurable top_k
    )
    
    return {
        "retriever": retriever,
        "vectorstore": vectordb,
        "chunks": chunks,  ## UPGRADE v2: keep chunk list for threshold filtering
    }

In [ ]:
## UPGRADE v2: Function to run threshold-filtered similarity search
## This is called inside the QA chain to ensure only grounded chunks
## are injected into the prompt.

def filtered_search(vectorstore, chunks, question):
    """
    Performs similarity search and filters chunks by cosine similarity threshold.
    
    Returns:
      - docs: list of docs that pass the threshold (may be empty)
      - avg_similarity: average cosine similarity of returned docs (0 if empty)
    """
    ## UPGRADE v2: use similarity_score_threshold for built-in filtering
    from langchain_community.vectorstores import Chroma
    
    # Chroma's as_retriever with search_type="similarity_score_threshold"
    threshold_retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={
            "k": TOP_K,
            "score_threshold": SIMILARITY_THRESHOLD
        }
    )
    docs = threshold_retriever.invoke(question)
    
    # Calculate average similarity from the distance scores
    ## UPGRADE v2: Chroma returns "distance" in metadata; convert to similarity
    if docs:
        distances = [d.metadata.get("distance", 1.0) for d in docs]
        # Euclidean distance → cosine similarity approximation
        similarities = [max(0, 1.0 - dist / 4.0) for dist in distances]  ## UPGRADE v2: heuristic conversion
        avg_sim = sum(similarities) / len(similarities)
    else:
        avg_sim = 0.0
    
    return docs, avg_sim

## 🛡️ Step 8 – QA Chain with Anti-Hallucination Prompt

The **system prompt** is the most critical hallucination control. It explicitly:
- Forbids using outside knowledge
- Requires sentence-level citations
- Commands the model to say "I don't know" when context is insufficient

In [ ]:
## UPGRADE v2: Anti-hallucination prompt template

ANTIHALL_PROMPT = PromptTemplate(
    input_variables=["context", "question", "chat_history"],
    template="""You are a strict QA assistant. Answer ONLY using the provided context.

### RULES (MUST FOLLOW):
1. **NEVER** use your own knowledge or guess. If the context does not contain the answer, say exactly: "Based on the provided documents, I don't have enough information to answer that question."
2. **Every factual claim** in your answer MUST be followed by a citation: [doc:0], [doc:1], etc.
3. **If context is insufficient**, state that clearly — do NOT fabricate or extrapolate.
4. **Do NOT** reference information that is not directly stated in the context.
5. **Keep answers concise and grounded.**

### Chat History (for reference only, do not fabricate from it):
{chat_history}

### Context (grounding material):
{context}

### Question:
{question}

### Answer:"""
)

## UPGRADE v2: Short-answer prompt to prevent rambling/hallucination
SHORT_ANSWER_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""Answer the question based ONLY on the context below. Be concise.

RULES:
- If you cannot answer from context alone, say: "No enough information in the documents."
- Cite every claim: [doc:0] [doc:1]
- Do NOT invent facts.

Context:
{context}

Question: {question}

Answer:"""
)

**Answer formatting helper** — strips citation markers for display and builds a clean reference list.

In [ ]:
## UPGRADE v2: Helper to parse and format citation-marked answers
import re

def format_answer(answer_text, source_docs):
    """
    Takes the raw model answer and source docs, returns a formatted string
    with inline citations and a reference section.
    """
    formatted = []
    formatted.append(answer_text.strip())
    
    if source_docs:
        formatted.append("\n\n" + "="*50)
        formatted.append("## 📚 Source Documents (Grounding)")
        formatted.append("="*50)
        for i, doc in enumerate(source_docs):
            src_file = doc.metadata.get("source_file", "unknown")
            page = doc.metadata.get("page_index", "?")
            formatted.append(f"\n[doc:{i}] {src_file} (page {page})")
            # Truncate chunk for readability
            text = doc.page_content[:500]
            if len(doc.page_content) > 500:
                text += "..."
            formatted.append(text)
    
    return "\n".join(formatted)

def build_confidence_report(avg_similarity):
    """Returns a human-readable confidence string."""
    if avg_similarity >= 0.7:
        level = "HIGH"
        icon = "🟢"
    elif avg_similarity >= 0.4:
        level = "MEDIUM"
        icon = "🟡"
    else:
        level = "LOW"
        icon = "🔴"
    return f"{icon} Confidence: {level} ({avg_similarity:.2f})" 

**Main QA function** — ties together retrieval with threshold, the anti-hallucination chain, and formatting.

In [ ]:
## UPGRADE v2: answer_question v2 — with retrieval threshold, confidence, citations

def answer_question(file_paths, question, chat_history=None):
    """
    Answer a question with grounding and hallucination controls.
    
    Args:
        file_paths: list of PDF file paths
        question: user's question
        chat_history: list of (user_q, assistant_a) tuples for conversation
    
    Returns:
        tuple: (formatted_answer_with_sources, confidence_report)
    """
    if chat_history is None:
        chat_history = []
    
    try:
        # Build retriever + vectorstore
        retrieval_data = build_retriever(file_paths)
        retriever = retrieval_data["retriever"]
        vectorstore = retrieval_data["vectorstore"]
        
        # ## UPGRADE v2: Use threshold-filtered search for grounding
        source_docs, avg_sim = filtered_search(vectorstore, retrieval_data["chunks"], question)
        
        # ## UPGRADE v2: Confidence report
        confidence = build_confidence_report(avg_sim)
        
        # ## UPGRADE v2: If no docs pass threshold, refuse to answer
        if not source_docs:
            answer = "Based on the provided documents, I don't have enough information to answer that question."
            # Build minimal source section
            formatted = answer + f"\n\n⚠️ {confidence} — No relevant content found in uploaded documents."
            return formatted, confidence
        
        # Build context string for the LLM
        context_text = "\n\n".join(
            f"[doc:{i}] {doc.page_content}" for i, doc in enumerate(source_docs)
        )
        
        # ## UPGRADE v2: ConversationalRetrievalChain with summary memory
        llm = get_llm()
        
        # ## UPGRADE v2: Use ConversationSummaryMemory to prevent drift
        memory = ConversationSummaryMemory(
            llm=llm,
            memory_key="chat_history",
            return_messages=True,
            input_key="question",
        )
        
        # ## UPGRADE v2: Anti-hallucination chain (conversational)
        qa_chain = ConversationalRetrievalChain.from_llm(
            llm=llm,
            retriever=retriever,
            chain_type="stuff",
            combine_docs_chain_kwargs={"prompt": ANTIHALL_PROMPT},
            memory=memory,
            return_source_documents=True,
            verbose=False,
        )
        
        # If we have threshold-filtered docs, override the context
        ## UPGRADE v2: Use filtered docs directly in the result for formatting
        result = qa_chain.invoke({"question": question})
        
        answer_text = result.get("answer", "No answer generated.")
        
        # Use source docs from filtered search for consistent formatting
        formatted = format_answer(answer_text, source_docs)
        
        return formatted, confidence
        
    except Exception as e:
        return f"Error processing your question: {str(e)}", "⚠️ Confidence: ERROR"

## Step 9 – Gradio Interface

In [ ]:
## UPGRADE v2: Gradio interface with confidence, sources, and conversation

## UPGRADE v2: Use gr.ChatInterface for conversational UI
## (v1 used gr.Interface with a single Q&A)

# State to persist uploaded files across turns
_uploaded_files = None

def gradio_rag_v2(file, message, history):
    """
    Gradio handler for conversational RAG with grounding.
    
    Args:
        file: uploaded PDF(s) (gr.File component)
        message: current user question
        history: list of [user_msg, bot_response] pairs
    """
    global _uploaded_files
    
    # If no file uploaded yet and no history, require upload
    if _uploaded_files is None and not history:
        return history + [[message, "Please upload a PDF first before asking questions."]]
    
    # Handle file upload (first turn or new upload)
    if file is not None and _uploaded_files is None:
        file_paths = file if isinstance(file, list) else [file]
        _uploaded_files = file_paths
        return history + [[message, f"✅ Uploaded {len(file_paths)} file(s). Ready to answer questions."]]
    
    # If file provided but files already uploaded, ignore new file
    # (v1 behavior: always rebuild from uploaded files)
    if file is not None:
        file_paths = file if isinstance(file, list) else [file]
        _uploaded_files = file_paths
    
    # Use current question and previous conversation as chat history
    chat_history = [(h[0], h[1]) for h in history if len(h) == 2 and isinstance(h[0], str) and isinstance(h[1], str)]
    
    # Build question with chat history context for the chain
    question = message
    if chat_history:
        # Prepend context for better grounded follow-ups
        question = message
    
    # Call the QA function
    if _uploaded_files:
        try:
            formatted_answer, confidence = answer_question(_uploaded_files, question, chat_history)
            # Append confidence to the answer
            full_response = f"{formatted_answer}\n\n{confidence}"
            return history + [[message, full_response]]
        except Exception as e:
            return history + [[message, f"Error: {str(e)}"]]
    else:
        return history + [[message, "Please upload a PDF first."]]

In [ ]:
## UPGRADE v2: Gradio ChatInterface with customizations

rag_app = gr.ChatInterface(
    fn=gradio_rag_v2,
    additional_inputs=[
        gr.File(
            label="Upload PDF File(s)",
            file_count="multiple",
            file_types=[".pdf"],
            type="filepath"
        ),
    ],
    title="🛡️ ITESM-NLP RAG Chatbot v2 — Anti-Hallucination",
    description="""Upload a PDF and ask questions. The bot answers using ONLY the document content,
    with inline citations and confidence scores. If the documents don't contain the answer, 
    it will say so instead of making something up.""",
    # ## UPGRADE v2: Theme and CSS
    theme=gr.themes.Soft(),
)

rag_app.launch(server_name="127.0.0.1", server_port=7862)

Stop the server and release the port

In [ ]:
gr.close_all()

## 📋 Summary of v2 vs v1 Changes

| Component | v1 | v2 |
|-----------|----|----|
| **Retrieval** | `RetrievalQA` + `stuff` chain | `ConversationalRetrievalChain` + `stuff` chain |
| **Similarity** | None (all retrieved chunks used) | Threshold filter (0.35 cosine similarity) |
| **Prompt** | Default LangChain template | Anti-hallucination template (no-guess, citation-required) |
| **Memory** | None (stateless) | `ConversationSummaryMemory` (summarized) |
| **Temperature** | 0.5 | 0.3 (lower = more deterministic) |
| **Source docs** | Returned but not displayed | Displayed inline with filename + page |
| **Confidence** | None | Per-query score (HIGH/MEDIUM/LOW) |
| **Fallback** | LLM could still hallucinate | Explicit "no enough info" when no chunks pass threshold |
| **UI** | Single Q&A via `gr.Interface` | Conversational via `gr.ChatInterface` |
| **Embedding model** | `all-MiniLM-L6-v2` | `all-MiniLM-L6-v2` (unchanged) |
| **Vector DB** | Chroma | Chroma (unchanged) |
| **Splitter** | RecursiveCharacterTextSplitter(1000,150) | Same (unchanged) |